Identificação dos arquivos e materialização das tabelas.

In [0]:
from pyspark.sql import functions as F

In [0]:
CAMINHO = "/Workspace/Users/livia.aguiarcc@gmail.com/turismo-bilateral-br-kr/dados/chegadas_brasil/*.csv"
TABELA = "chegadas.brasil.bronze"

Leitura das tabelas.

In [0]:
bruto = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "latin1")
    .option("mode","PERMISSIVE")
    .load(CAMINHO)
)

print("Colunas lidas do arquivo: ")
for c in bruto.columns:
    print(f"  {c!r}")

Formatação do nome das colunas - remoção do espaço.

In [0]:
RENOMEAR = {
    "Continente": "continente",
    "cod continente": "cod_continente",
    "País": "pais",
    "cod pais": "cod_pais",
    "UF": "uf",
    "cod uf": "cod_uf",
    "Via": "via",
    "cod via": "cod_via",
    "ano": "ano",
    "Mês": "mes",
    "cod mes": "cod_mes",
    "Chegadas": "chegadas"
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Colunas faltando: {faltando}"

renomeado = bruto.select(
    *[
        F.col(f"`{origem}`").cast("string").alias(novo)
        for origem, novo in RENOMEAR.items()
    ],
    F.col("_metadata.file_name").alias("_arquivo_origem")
)

Auditoria: colunas de origem e metadatos!

In [0]:
bronze = renomeado.withColumn(
    "ingerido_em", F.current_timestamp()
)

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS turismo_bilateral")
spark.sql("CREATE SCHEMA IF NOT EXISTS turismo_bilateral.bronze")

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS chegadas")
spark.sql("CREATE SCHEMA IF NOT EXISTS chegadas.brasil")

(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - Chegadas Internacionais no Brasil (2018 a 2025).
     Dado bruto: todas as colunas string, nenhuma linha descartada.
     Carga full refresh idempotente a partir de turismo-bilateral-br-kr/dados/chegadas_brasil.'
""")

In [0]:
display(
    spark.sql(f"""
        SELECT _arquivo_origem, COUNT(*) AS linhas, MAX(ingerido_em) AS ingerido_em
        FROM {TABELA}
        GROUP BY _arquivo_origem
        ORDER BY _arquivo_origem
    """)
)